In [ ]:
from pathlib import Path
import os, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 20260819
random.seed(SEED); np.random.seed(SEED)
warnings.filterwarnings('ignore')
DATA_ROOT = Path('TERA Analysis')
PROJECT_ROOT = Path('TERA')
RESULTS = PROJECT_ROOT / 'results'; FIGURES = PROJECT_ROOT / 'figures'
RESULTS.mkdir(exist_ok=True); FIGURES.mkdir(exist_ok=True)
print('Data:', DATA_ROOT)
print('Outputs:', PROJECT_ROOT)


In [ ]:
import tensorflow as tf
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedGroupKFold, GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
tf.random.set_seed(SEED)
df=pd.read_csv(DATA_ROOT/'exploratory_analysis/weekly/tera_weekly.csv').dropna(subset=['USUBJID','is_adherent_wk']).reset_index(drop=True)
df['USUBJID']=df.USUBJID.astype(int); y=df.is_adherent_wk.astype(int).to_numpy(); groups=df.USUBJID.to_numpy()
per_week=['weekend_adherent','epoch_time1_mode_agg','time_mean','time_std','is_adherent_wk']
dynamic_columns=[f't-{lag} {feature}' for lag in range(4,0,-1) for feature in per_week]
current={'USUBJID','is_adherent_wk','week_num','weekend_adherent','time_mean','time_std','adherent_percent','epoch_time1_mode_agg'}
static_columns=[c for c in df.columns if c not in current and c not in dynamic_columns]
Xd=df[dynamic_columns].apply(pd.to_numeric,errors='coerce').to_numpy(float).reshape(-1,4,len(per_week)); Xs=df[static_columns].copy()
print(len(df),df.USUBJID.nunique(),Xd.shape,Xs.shape)


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix

def classification_metrics(y_true, probability, threshold=0.5):
    prediction = (np.asarray(probability) >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, prediction, labels=[0,1]).ravel()
    return {
        'accuracy_pct': 100*accuracy_score(y_true,prediction),
        'precision_pct': 100*precision_score(y_true,prediction,zero_division=0),
        'recall_pct': 100*recall_score(y_true,prediction,zero_division=0),
        'specificity_pct': 100*tn/(tn+fp) if tn+fp else np.nan,
    }


In [ ]:
def build_model(steps,n_dynamic,n_static=0):
    dynamic_input=tf.keras.Input((steps,n_dynamic),name='dynamic')
    x=tf.keras.layers.LSTM(128,return_sequences=True)(dynamic_input)
    x=tf.keras.layers.LSTM(128)(x); inputs=[dynamic_input]
    if n_static:
        static_input=tf.keras.Input((n_static,),name='static')
        static_branch=tf.keras.layers.Dense(32,activation='relu')(static_input)
        x=tf.keras.layers.Concatenate()([x,static_branch]); inputs.append(static_input)
    x=tf.keras.layers.Dropout(.25)(x); x=tf.keras.layers.Dense(32,activation='relu')(x)
    output=tf.keras.layers.Dense(1,activation='sigmoid')(x)
    model=tf.keras.Model(inputs,output)
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-4),loss='binary_crossentropy')
    return model


In [ ]:
outer=StratifiedGroupKFold(5,shuffle=True,random_state=SEED); probability=np.full(len(df),np.nan)
for fold,(train_all,test) in enumerate(outer.split(np.zeros(len(y)),y,groups),1):
    train_rel,val_rel=next(GroupShuffleSplit(1,test_size=.20,random_state=SEED+fold).split(train_all,y[train_all],groups[train_all])); train=train_all[train_rel]; val=train_all[val_rel]
    flat=Xd.reshape(len(df),-1); imp=SimpleImputer(strategy='median').fit(flat[train]); scale=StandardScaler().fit(imp.transform(flat[train]))
    d=lambda ix: scale.transform(imp.transform(flat[ix])).reshape(-1,4,len(per_week))
    cats=[c for c in static_columns if Xs[c].dtype=='object']; nums=[c for c in static_columns if c not in cats]
    pre=ColumnTransformer([('num',Pipeline([('imp',SimpleImputer(strategy='median')),('sc',StandardScaler())]),nums),('cat',Pipeline([('imp',SimpleImputer(strategy='most_frequent')),('oh',OneHotEncoder(handle_unknown='ignore',sparse_output=False))]),cats)]).fit(Xs.iloc[train])
    strn,sval,stest=[pre.transform(Xs.iloc[ix]).astype('float32') for ix in [train,val,test]]
    n1=y[train].sum(); n0=len(train)-n1; weights={0:len(train)/(2*n0),1:len(train)/(2*n1)}
    tf.keras.backend.clear_session(); model=build_model(4,len(per_week),strn.shape[1]); stop=tf.keras.callbacks.EarlyStopping(monitor='val_loss',patience=3,restore_best_weights=True)
    model.fit([d(train),strn],y[train],validation_data=([d(val),sval],y[val]),epochs=30,batch_size=64,class_weight=weights,callbacks=[stop],verbose=0)
    probability[test]=model.predict([d(test),stest],verbose=0).ravel()
metrics=classification_metrics(y,probability); display(pd.DataFrame([metrics])); pd.DataFrame([metrics]).to_csv(RESULTS/'weekly_integrated_metrics.csv',index=False)
pd.DataFrame({'USUBJID':groups,'outcome':y,'probability':probability}).to_csv(RESULTS/'weekly_integrated_predictions.csv',index=False)
